# v2 Smoke — Blender + Flux Background Inpaint

Interactive smoke на RunPod A100/H100. Класс: **light_vehicle** (GAZ Tigr).

**Flow:** 1 frame → eyeball **Gate 1** → 5 frames → **Gate 2** → 20 frames batch → zip download.

**Контракт:** vehicle pixels байт-ідентичні після composite, bbox unchanged.

In [ ]:
# Cell 1 — env check
import os, sys, torch
print(f'Python: {sys.version.split()[0]}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'  CUDA: {torch.version.cuda}')
assert torch.cuda.is_available(), 'no CUDA — wrong pod template?'
assert torch.cuda.get_device_properties(0).total_memory >= 40e9, 'need >=40GB VRAM'

assert 'HF_TOKEN' in os.environ, "HF_TOKEN env var required (RunPod 'Pod env vars')"
print(f"HF_TOKEN: {os.environ['HF_TOKEN'][:8]}...")
print(f"HF_HOME: {os.environ.get('HF_HOME', '<default>')}")

from huggingface_hub import whoami
hf_user = whoami(token=os.environ['HF_TOKEN'])['name']
print(f'HF user: {hf_user}')
print('\n✓ env ready')

In [ ]:
# Cell 2 — repo + imports
import subprocess, sys
from pathlib import Path

REPO_DIR = Path('/workspace/yolo-bluebierd')
assert REPO_DIR.exists(), f'clone repo first: git clone <url> {REPO_DIR}'

subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=False)
subprocess.run(['pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements_diffusion.txt')], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from datasetforge.pipelines.inpaint import bg_filler, composite, prompts
print('✓ modules imported')

In [ ]:
# Cell 3 — Stage 1: render 1 frame at diffusion resolution
import os, subprocess, yaml
from pathlib import Path

REPO_DIR = Path('/workspace/yolo-bluebierd')
ASSETS = REPO_DIR / 'datasetforge/assets'
CFG_SRC = REPO_DIR / 'datasetforge/configs/v1_light_vehicle.yaml'
OUT = Path('/workspace/output/v2_smoke_1frame')
OUT.mkdir(parents=True, exist_ok=True)

cfg_dict = yaml.safe_load(CFG_SRC.read_text())
inf_size = cfg_dict.get('image_size_diffusion', [1024, 1024])
cfg_dict['image_size'] = inf_size
cfg_dict['diffusion']['enabled'] = True

CFG_RUN = OUT / 'v1_light_vehicle_diff.yaml'
CFG_RUN.write_text(yaml.safe_dump(cfg_dict))
print(f'render @ {inf_size}, cfg → {CFG_RUN}')

env = os.environ.copy(); env['MPLBACKEND'] = 'agg'
cmd = ['blenderproc', 'run', str(REPO_DIR / 'datasetforge/engine/render_runner.py'),
       '--config', str(CFG_RUN), '--n', '1', '--out', str(OUT),
       '--assets-root', str(ASSETS), '--seed', '42']
subprocess.run(cmd, env=env, check=True)

print(f'\n✓ Stage 1 → {OUT}')
for sub in ['images','labels','depth','normals','vehicle_masks','metadata']:
    n = len(list((OUT / sub).glob('*')))
    print(f'  {sub}: {n} file(s)')

In [ ]:
# Cell 4 — Stage 1 preview (RGB+bbox / depth / normals / mask)
import json, cv2, numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path

OUT = Path('/workspace/output/v2_smoke_1frame')
stem = sorted(p.stem for p in (OUT / 'images').glob('*.jpg'))[0]
print(f'preview: {stem}')

rgb = np.array(Image.open(OUT/'images'/f'{stem}.jpg'))
depth = cv2.imread(str(OUT/'depth'/f'{stem}.png'), cv2.IMREAD_UNCHANGED)
normals_raw = cv2.imread(str(OUT/'normals'/f'{stem}.png'), cv2.IMREAD_UNCHANGED)
mask = cv2.imread(str(OUT/'vehicle_masks'/f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
meta = json.loads((OUT/'metadata'/f'{stem}.json').read_text())

H, W = rgb.shape[:2]
rgb_pil = Image.fromarray(rgb).copy()
d = ImageDraw.Draw(rgb_pil)
lbl = OUT / 'labels' / f'{stem}.txt'
if lbl.exists():
    for line in lbl.read_text().strip().splitlines():
        p = line.split(); xc, yc, w, h = map(float, p[1:5])
        d.rectangle([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H], outline='lime', width=3)

normals_viz = ((normals_raw.astype(np.float32)/65535.0)*255).astype(np.uint8) if normals_raw.dtype==np.uint16 else normals_raw

fig, ax = plt.subplots(2, 2, figsize=(12, 12))
ax[0,0].imshow(rgb_pil); ax[0,0].set_title('RGB + bbox'); ax[0,0].axis('off')
ax[0,1].imshow(depth, cmap='viridis'); ax[0,1].set_title(f'Depth (mm) max={int(depth.max())}'); ax[0,1].axis('off')
ax[1,0].imshow(normals_viz); ax[1,0].set_title('Normals (encoded)'); ax[1,0].axis('off')
ax[1,1].imshow(mask, cmap='gray'); ax[1,1].set_title(f'Mask veh_px={int((mask>=128).sum())}'); ax[1,1].axis('off')
plt.tight_layout(); plt.show()

print(f"\nsun: {meta['sun_cardinal']} @ {meta['sun_elevation_deg']:.0f}° elev")
print(f"season={meta['season']} weather={meta['weather']} landscape={meta['landscape']}")
print(f"distance={meta['distance_m']}m angle={meta['view_angle_deg']}°")

In [ ]:
# Cell 5 — Load Flux pipeline (one-time, ~60-90s, ~30 GB VRAM)
import time, yaml, torch
from pathlib import Path
from datasetforge.pipelines.inpaint import bg_filler

CFG_RUN = Path('/workspace/output/v2_smoke_1frame/v1_light_vehicle_diff.yaml')
cfg_full = yaml.safe_load(CFG_RUN.read_text())
diff_cfg = cfg_full['diffusion']
print(f"pipeline: {diff_cfg['pipeline']}")
print(f"base:     {diff_cfg['base_model']}")
print(f"depth CN: {diff_cfg['depth_controlnet']}")
print(f"steps={diff_cfg['steps']} guidance={diff_cfg['guidance']} cn_scale={diff_cfg['controlnet_scale']}")

t0 = time.time()
pipe = bg_filler.load_pipeline(diff_cfg, device='cuda')
torch.cuda.synchronize()
print(f'\n✓ loaded in {time.time()-t0:.1f}s')
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f}/{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# Cell 6 — Stage 3: inpaint 1 frame
import time
from pathlib import Path
from datasetforge.pipelines.inpaint import bg_filler

OUT = Path('/workspace/output/v2_smoke_1frame')
stem = sorted(p.stem for p in (OUT/'images').glob('*.jpg'))[0]
(OUT/'ai_bg').mkdir(exist_ok=True)

t0 = time.time()
sidecar = bg_filler.inpaint_one(
    pipe,
    rgb_path=OUT/'images'/f'{stem}.jpg',
    depth_path=OUT/'depth'/f'{stem}.png',
    mask_path=OUT/'vehicle_masks'/f'{stem}.png',
    meta_path=OUT/'metadata'/f'{stem}.json',
    out_path=OUT/'ai_bg'/f'{stem}.png',
    diffusion_cfg=diff_cfg,
)
print(f'\n✓ inpaint {time.time()-t0:.1f}s | seed={sidecar["seed"]}')
print(f'\nprompt: {sidecar["prompt"]}')

In [ ]:
# Cell 7 — Stage 4: composite + 3-up preview
import numpy as np, cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path
from datasetforge.pipelines.inpaint import composite

OUT = Path('/workspace/output/v2_smoke_1frame')
stem = sorted(p.stem for p in (OUT/'images').glob('*.jpg'))[0]
(OUT/'composite').mkdir(exist_ok=True)

stats = composite.composite_one(
    rgb_path=OUT/'images'/f'{stem}.jpg',
    ai_bg_path=OUT/'ai_bg'/f'{stem}.png',
    mask_path=OUT/'vehicle_masks'/f'{stem}.png',
    normals_path=OUT/'normals'/f'{stem}.png',
    meta_path=OUT/'metadata'/f'{stem}.json',
    out_path=OUT/'composite'/f'{stem}.png',
    diffusion_cfg=diff_cfg,
    assert_pixel_identity=True,
)
print(f'✓ composite stats: {stats}')

rgb = np.array(Image.open(OUT/'images'/f'{stem}.jpg'))
ai_bg = np.array(Image.open(OUT/'ai_bg'/f'{stem}.png'))
comp = np.array(Image.open(OUT/'composite'/f'{stem}.png'))
mask = cv2.imread(str(OUT/'vehicle_masks'/f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
mb = mask >= 128
diff = int(np.abs(rgb.astype(int) - comp.astype(int))[mb].max())
print(f'vehicle pixel max-diff: {diff} (expect 0 with relight OFF)')

H, W = comp.shape[:2]
comp_pil = Image.fromarray(comp).copy()
d = ImageDraw.Draw(comp_pil)
lbl = OUT/'labels'/f'{stem}.txt'
if lbl.exists():
    for line in lbl.read_text().strip().splitlines():
        p = line.split(); xc, yc, w, h = map(float, p[1:5])
        d.rectangle([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H], outline='lime', width=3)

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(rgb); ax[0].set_title('Stage 1 — raw 3D'); ax[0].axis('off')
ax[1].imshow(ai_bg); ax[1].set_title('Stage 3 — AI background'); ax[1].axis('off')
ax[2].imshow(comp_pil); ax[2].set_title(f'Stage 4 — composite (diff={diff})'); ax[2].axis('off')
plt.tight_layout(); plt.show()

## ⚖️ APPROVE GATE 1

Подивись на composite вище:
- ✓ Vehicle pixels не торкнуті (diff=0)?
- ✓ Bbox tight навколо vehicle?
- ✓ AI background виглядає realistic для season+landscape?
- ✓ Тіні vehicle vs ai-bg відносно узгоджені?

**Якщо ОК** → run Cell 9 (5 frames).

**Якщо НІ** — підкрутити у `diff_cfg`:
- AI bg тіні з не з тієї сторони → `controlnet_scale` 0.4 → 0.6 → 0.8 sweep
- vehicle floats / геометрія губиться → `controlnet_scale` 0.8
- AI bg занадто blurred → `steps` 40-50
- AI bg unrealistic / оверсатурований → `guidance` 20-25

Re-run cells 6-7.

In [ ]:
# Cell 9 — Scale до 5 frames (4 seasons × landscapes)
import time, subprocess, os
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from datasetforge.pipelines.inpaint import bg_filler, composite

REPO_DIR = Path('/workspace/yolo-bluebierd')
OUT5 = Path('/workspace/output/v2_smoke_5frame')
OUT5.mkdir(parents=True, exist_ok=True)
CFG = Path('/workspace/output/v2_smoke_1frame/v1_light_vehicle_diff.yaml')

env = os.environ.copy(); env['MPLBACKEND'] = 'agg'
subprocess.run(['blenderproc', 'run',
                str(REPO_DIR/'datasetforge/engine/render_runner.py'),
                '--config', str(CFG), '--n', '5', '--out', str(OUT5),
                '--assets-root', str(REPO_DIR/'datasetforge/assets'),
                '--seed', '100'], env=env, check=True)

(OUT5/'ai_bg').mkdir(exist_ok=True); (OUT5/'composite').mkdir(exist_ok=True)
stems = sorted(p.stem for p in (OUT5/'images').glob('*.jpg'))
t0 = time.time()
for stem in stems:
    bg_filler.inpaint_one(pipe,
        rgb_path=OUT5/'images'/f'{stem}.jpg', depth_path=OUT5/'depth'/f'{stem}.png',
        mask_path=OUT5/'vehicle_masks'/f'{stem}.png', meta_path=OUT5/'metadata'/f'{stem}.json',
        out_path=OUT5/'ai_bg'/f'{stem}.png', diffusion_cfg=diff_cfg)
    composite.composite_one(
        rgb_path=OUT5/'images'/f'{stem}.jpg', ai_bg_path=OUT5/'ai_bg'/f'{stem}.png',
        mask_path=OUT5/'vehicle_masks'/f'{stem}.png', normals_path=OUT5/'normals'/f'{stem}.png',
        meta_path=OUT5/'metadata'/f'{stem}.json', out_path=OUT5/'composite'/f'{stem}.png',
        diffusion_cfg=diff_cfg, assert_pixel_identity=True)
print(f'\n✓ 5-frame Stage 3+4 in {time.time()-t0:.1f}s')

fig, ax = plt.subplots(5, 3, figsize=(15, 25))
for r, stem in enumerate(stems):
    ax[r,0].imshow(np.array(Image.open(OUT5/'images'/f'{stem}.jpg'))); ax[r,0].set_title(f'{stem} raw')
    ax[r,1].imshow(np.array(Image.open(OUT5/'ai_bg'/f'{stem}.png'))); ax[r,1].set_title(f'{stem} ai_bg')
    ax[r,2].imshow(np.array(Image.open(OUT5/'composite'/f'{stem}.png'))); ax[r,2].set_title(f'{stem} composite')
    for c in range(3): ax[r,c].axis('off')
plt.tight_layout(); plt.show()

## ⚖️ APPROVE GATE 2

5 кадрів — різні seasons / landscapes / GAZ Tigr варіанти. Перевір:
- ✓ Кожен composite має vehicle pixels intact (assert_pixel_identity не впав)?
- ✓ Сезони виглядають **по-різному**?
- ✓ Жодних AI-tells: melted text, impossible architecture, vehicle 'floats', tiling artifacts?
- ✓ Тіні +/- кореляють з `sun_cardinal` у metadata?

**Якщо ОК** → Cell 11 (20-frame batch + regression test).

In [ ]:
# Cell 11 — 20-frame batch
import time, os, subprocess
from pathlib import Path
from datasetforge.pipelines.inpaint import bg_filler, composite

REPO_DIR = Path('/workspace/yolo-bluebierd')
OUT20 = Path('/workspace/output/v2_smoke_20frame')
OUT20.mkdir(parents=True, exist_ok=True)
CFG = Path('/workspace/output/v2_smoke_1frame/v1_light_vehicle_diff.yaml')

env = os.environ.copy(); env['MPLBACKEND'] = 'agg'
t_wall = time.time()
subprocess.run(['blenderproc', 'run',
                str(REPO_DIR/'datasetforge/engine/render_runner.py'),
                '--config', str(CFG), '--n', '20', '--out', str(OUT20),
                '--assets-root', str(REPO_DIR/'datasetforge/assets'),
                '--seed', '200'], env=env, check=True)

(OUT20/'ai_bg').mkdir(exist_ok=True); (OUT20/'composite').mkdir(exist_ok=True)
stems = sorted(p.stem for p in (OUT20/'images').glob('*.jpg'))
for stem in stems:
    bg_filler.inpaint_one(pipe,
        rgb_path=OUT20/'images'/f'{stem}.jpg', depth_path=OUT20/'depth'/f'{stem}.png',
        mask_path=OUT20/'vehicle_masks'/f'{stem}.png', meta_path=OUT20/'metadata'/f'{stem}.json',
        out_path=OUT20/'ai_bg'/f'{stem}.png', diffusion_cfg=diff_cfg)
    composite.composite_one(
        rgb_path=OUT20/'images'/f'{stem}.jpg', ai_bg_path=OUT20/'ai_bg'/f'{stem}.png',
        mask_path=OUT20/'vehicle_masks'/f'{stem}.png', normals_path=OUT20/'normals'/f'{stem}.png',
        meta_path=OUT20/'metadata'/f'{stem}.json', out_path=OUT20/'composite'/f'{stem}.png',
        diffusion_cfg=diff_cfg, assert_pixel_identity=True)

wall = time.time() - t_wall
GPU_USD_PER_HR = 1.7  # A100 80GB; H100 ≈ 2.5
print(f'\n✓ 20 frames in {wall:.0f}s ({wall/60:.1f} min)')
print(f'  est cost: ${wall/3600 * GPU_USD_PER_HR:.3f}')

In [ ]:
# Cell 12 — pack + download
import shutil
from pathlib import Path

OUT20 = Path('/workspace/output/v2_smoke_20frame')
zip_path = shutil.make_archive('/workspace/output/v2_smoke_20frame', 'zip',
                                root_dir=OUT20.parent, base_dir=OUT20.name)
size_mb = Path(zip_path).stat().st_size / 1e6
print(f'✓ {zip_path} ({size_mb:.1f} MB)')
print(f'\nDownload via RunPod web → File Browser → /workspace/output/v2_smoke_20frame.zip')

In [ ]:
# Cell 13 — pixel-precision regression test + co-occurrence sanity
import numpy as np, cv2, json
from PIL import Image
from pathlib import Path
from collections import Counter

OUT20 = Path('/workspace/output/v2_smoke_20frame')
stems = sorted(p.stem for p in (OUT20/'images').glob('*.jpg'))
print(f'validating {len(stems)} frames')

failures = []
for stem in stems:
    rgb = np.array(Image.open(OUT20/'images'/f'{stem}.jpg'))
    comp = np.array(Image.open(OUT20/'composite'/f'{stem}.png'))
    mask = cv2.imread(str(OUT20/'vehicle_masks'/f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
    mb = mask >= 128
    diff = int(np.abs(rgb.astype(int) - comp.astype(int))[mb].max()) if mb.any() else 0
    if diff != 0:
        failures.append((stem, diff))

if failures:
    print(f'❌ {len(failures)} frames violated pixel-freeze contract:')
    for s, d in failures[:5]: print(f'  {s}: max diff {d}')
else:
    print(f'✓ all {len(stems)} frames: vehicle pixels byte-identical')

pairs = Counter()
for stem in stems:
    m = json.loads((OUT20/'metadata'/f'{stem}.json').read_text())
    pairs[(m['season'], m['landscape'])] += 1
print(f'\n(season, landscape) distribution:')
for pair, n in pairs.most_common():
    pct = 100*n/len(stems)
    mark = '⚠' if pct > 30 else ' '
    print(f'  {mark} {pair}: {n} ({pct:.0f}%)')

## 🛑 Cleanup reminder

Pod продовжує білингитись до явної зупинки.

```bash
runpodctl pod stop <POD_ID>
```

Або через web UI → Pods → Stop.

**Після smoke — ротейтнути RunPod API key** (Settings → API Keys → Revoke + Create new).
**Після smoke — ротейтнути HF token** якщо хочеш бути параноїдальною.